# 第44章：欧式期权定价与对冲教学

Yuxiao Ma

全部为SIMULATED/CONSTRUCTED。默认新算机制和小型MC，保存三幅新图；无行情或训练，不等于完整CLI协议。时间按年，mu为含股息总收益漂移，P股价漂移mu−q、Q漂移r−q。直接1×12对冲路径与CLI先画大数组再聚合的随机数消费不同，同seed不代表同一条历史路径。

[本语言逐本运行模式与准备条件](../NOTEBOOK_GUIDE.md)。运行步骤、数据要求和预期输出见本指南。

In [ ]:
import os, sys
from pathlib import Path
from uuid import uuid4
candidate = Path.cwd().resolve()
ROOT = next((p for p in (candidate, *candidate.parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start from the reader package root or a subdirectory")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("ch44_stochastic_derivatives_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
workspace_root = ROOT
print("Run directory:", RUN_DIR.relative_to(ROOT))
import numpy as np
import matplotlib.pyplot as plt
from quant_course.stochastic_calculus import brownian_increments, ito_left, quadratic_variation, gbm_exact, gbm_euler
from quant_course.derivatives_pricing import bs_price, bs_greeks, crr_price, mc_price, delta_hedge
p=dict(S=100.,K=100.,T=1.,r=.05,q=.02,sigma=.2)
print("SIMULATED", {k:bs_price(**p,kind=k) for k in ["call","put"]})

## 左点Itô积分与二次变差
有限网格I=0.5(W_T²−QV)，极限替换QV为T。有限路径偏差不应强改为零。

In [ ]:
dw=brownian_increments(16,1024,1.,4402)
w=np.column_stack([np.zeros(16),np.cumsum(dw,axis=1)])
qv=quadratic_variation(w); integ=ito_left(w,w[:,:-1])
assert np.allclose(integ,.5*(w[:,-1]**2-qv),atol=1e-12)
print("finite identity max residual",max(abs(integ-.5*(w[:,-1]**2-qv))))
fig,ax=plt.subplots(figsize=(8,3)); ax.plot(np.linspace(0,1,1025),w[:4].T)
ax.set(xlabel="Time (years)",ylabel="W (sqrt(years))",title="SIMULATED Brownian paths")
plt.savefig(RUN_DIR / "ch44_stochastic_derivatives_cell03.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close(); plt.close(fig)

## 共同增量：GBM精确与Euler
强误差比较同一布朗终值；不同噪声的路径差包含抽样变动。

In [ ]:
dw2=brownian_increments(128,1024,1.,4403)
rows=[]
for steps in [16,64,256,1024]:
    d=dw2.reshape(128,steps,1024//steps).sum(axis=2)
    ex=gbm_exact(100.,.06,.2,1/steps,d)
    eu=gbm_euler(100.,.06,.2,1/steps,d)
    err=eu[:,-1]-ex[:,-1]
    rows.append((steps,np.sqrt(np.mean(err**2)),np.mean(err),np.std(err,ddof=1)/np.sqrt(128)))
print("steps,strong RMSE,paired weak mean,paired SE",rows)
fig,ax=plt.subplots(); ax.loglog([x[0] for x in rows],[x[1] for x in rows],"o-")
ax.set(xlabel="Steps",ylabel="Terminal strong RMSE (currency/share)",title="SIMULATED common noise")
plt.savefig(RUN_DIR / "ch44_stochastic_derivatives_cell05.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close(); plt.close(fig)

## CRR与BS、Q下Monte Carlo
树有奇偶振荡。对偶样本SE以配对平均计，95%区间为正态近似；一次覆盖不证明算法正确。

In [ ]:
bs=bs_price(**p)
tree=[(s,crr_price(**p,steps=s)-bs) for s in [10,25,50,51,100,101,200,400]]
m=mc_price(**p,pairs=5000,seed=4401,antithetic=True)
print("CRR minus BS",tree); print("Q MC",m)
assert abs(bs_price(**p)-bs_price(**p,kind="put")-100*np.exp(-.02)+100*np.exp(-.05))<1e-10
fig,ax=plt.subplots(); ax.plot(*zip(*tree),"o-"); ax.axhline(0,color="gray")
ax.set(xlabel="Tree steps",ylabel="CRR minus BS (currency/share)",title="CONSTRUCTED tree convergence")
plt.savefig(RUN_DIR / "ch44_stochastic_derivatives_cell07.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close(); plt.close(fig)

## Greeks单位
Theta为日历时间导数（每年），Vega为单位波动率导数；1个百分点的近似金额变化乘.01。到期/零波动Greeks范围见函数文档。

In [ ]:
for spot in [80.,100.,120.]:
    for kind in ["call","put"]:
        a=dict(p,S=spot); g=bs_greeks(**a,kind=kind)
        h=1e-4
        fd=(bs_price(**dict(a,sigma=.2+h),kind=kind)-bs_price(**dict(a,sigma=.2-h),kind=kind))/(2*h)
        assert abs(fd-g["vega"])<1e-4
        print(spot,kind,g,"vega 1pp",.01*g["vega"])


## P下离散Delta对冲
左点持仓产生区间收益；同网格观察与瞬时成交是理想模拟约定，q现金股息采用协议中的左点近似。初始及清仓也计单边费用。

In [ ]:
dw3=brownian_increments(1,12,1.,4404)
path=gbm_exact(100.,.06,.2,1/12,dw3)[0]
for fee in [0.,.001]:
    hedge=delta_hedge(path,1.,.05,.02,.2,100.,cost_rate=fee)
    print("fee",fee,"portfolio minus payoff",hedge["error"],"cash cost",hedge["total_cost"])
    print("first and final ledger",hedge["ledger"][0],hedge["ledger"][-1])

## 时间反例与模型判断
改变未来路径不改变过去的左点积分。数组形状检查不能证明被积过程适应性；调用者必须证明信息只用当时已知部分。

In [ ]:
future=w.copy(); future[:,513:]+=3.
past1=ito_left(w[:,:513],w[:,:512]); past2=ito_left(future[:,:513],future[:,:512])
assert np.array_equal(past1,past2)
# Anticipating integrand W_T uses future information: E[W_T*W_T]=T, not zero.
anticipating=ito_left(w,np.repeat(w[:,-1,None],1024,axis=1))
assert np.allclose(anticipating,w[:,-1]**2)
print("prefix invariant",True,"anticipating path0",anticipating[0])